In [2]:
print('Ritu1')

Ritu1


In [3]:
from langchain_ai21.chat_models import ChatAI21
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_core.messages import SystemMessage, BaseMessage, HumanMessage, AIMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from typing import TypedDict, Annotated, List, Dict
from dotenv import load_dotenv
import json

load_dotenv()

# Initialize model and tools
model = ChatAI21(model='jamba-mini-1.7-2025-07')
searchTool = TavilySearchResults(max_results=3)
tools = [searchTool]
model_with_tools = model.bind_tools(tools)

# System prompt for Step 1: Content Outline Generation
OUTLINE_SYSTEM_PROMPT = SystemMessage(
    content="""
You are an expert presentation designer and content strategist.

Your task: Create a STRUCTURED OUTLINE for a PowerPoint presentation.

Output format (JSON):
{
  "title": "Main presentation title",
  "total_slides": number,
  "slides": [
    {
      "slide_number": 1,
      "slide_title": "Title of the slide",
      "key_points": ["Point 1", "Point 2", "Point 3"],
      "content_type": "introduction/explanation/comparison/conclusion"
    }
  ]
}

Rules:
- Create a logical flow from introduction to conclusion
- Each slide should have 3-5 key points
- Be specific about what each slide will cover
- Ensure comprehensive coverage of the topic
- Use tools to research if needed for accuracy
- Return ONLY valid JSON, no additional text
"""
)

# System prompt for Step 2: Detailed Content Generation
DETAIL_SYSTEM_PROMPT = SystemMessage(
    content="""
You are an expert content writer for presentations.

Your task: Generate DETAILED, ENGAGING content for a specific slide.

For each key point:
- Provide 2-3 sentences of explanation
- Include relevant examples, statistics, or facts
- Make it clear, concise, and presentation-ready
- Use simple language that's easy to understand

Output format:
{
  "slide_number": number,
  "slide_title": "Title",
  "detailed_content": [
    {
      "key_point": "Main point",
      "explanation": "Detailed explanation (2-3 sentences)",
      "visual_suggestion": "Suggestion for visual/diagram"
    }
  ],
  "speaker_notes": "Additional context for presenter"
}

Rules:
- Write presentation-ready content (not just bullet points)
- Include practical examples where relevant
- Use tools to get current data if needed
- Return ONLY valid JSON, no additional text
"""
)


class PptState(TypedDict):
    messages: Annotated[List[BaseMessage], add_messages]
    outline: Dict
    detailed_slides: List[Dict]
    current_slide_index: int
    topic: str
    num_slides: int


def chat_node(state: PptState):
    """Main chat node that processes messages"""
    messages = state['messages']
    result = model_with_tools.invoke(messages)
    return {'messages': result}


def generate_outline_node(state: PptState):
    """Step 1: Generate presentation outline"""
    topic = state['topic']
    num_slides = state['num_slides']
    
    prompt = HumanMessage(
        content=f"Create a {num_slides}-slide presentation outline on: {topic}"
    )
    
    messages = [OUTLINE_SYSTEM_PROMPT, prompt]
    result = model_with_tools.invoke(messages)
    
    # Parse the outline from the response
    try:
        outline_text = result.content
        # Extract JSON from response (handling potential markdown code blocks)
        if "```json" in outline_text:
            outline_text = outline_text.split("```json")[1].split("```")[0].strip()
        elif "```" in outline_text:
            outline_text = outline_text.split("```")[1].split("```")[0].strip()
        
        outline = json.loads(outline_text)
    except json.JSONDecodeError:
        # Fallback: create a basic structure
        outline = {
            "title": topic,
            "total_slides": num_slides,
            "slides": []
        }
    
    return {
        'messages': [result],
        'outline': outline,
        'current_slide_index': 0
    }


def generate_slide_detail_node(state: PptState):
    """Step 2: Generate detailed content for current slide"""
    outline = state['outline']
    current_index = state['current_slide_index']
    
    if current_index >= len(outline.get('slides', [])):
        return {'detailed_slides': state.get('detailed_slides', [])}
    
    current_slide = outline['slides'][current_index]
    
    prompt = HumanMessage(
        content=f"""Generate detailed content for this slide:
Slide Number: {current_slide['slide_number']}
Title: {current_slide['slide_title']}
Key Points: {', '.join(current_slide['key_points'])}
Content Type: {current_slide['content_type']}

Provide comprehensive, presentation-ready content."""
    )
    
    messages = [DETAIL_SYSTEM_PROMPT, prompt]
    result = model_with_tools.invoke(messages)
    
    # Parse the detailed content
    try:
        detail_text = result.content
        if "```json" in detail_text:
            detail_text = detail_text.split("```json")[1].split("```")[0].strip()
        elif "```" in detail_text:
            detail_text = detail_text.split("```")[1].split("```")[0].strip()
        
        detailed_slide = json.loads(detail_text)
    except json.JSONDecodeError:
        detailed_slide = {
            "slide_number": current_slide['slide_number'],
            "slide_title": current_slide['slide_title'],
            "detailed_content": [],
            "raw_response": result.content
        }
    
    detailed_slides = state.get('detailed_slides', [])
    detailed_slides.append(detailed_slide)
    
    return {
        'messages': [result],
        'detailed_slides': detailed_slides,
        'current_slide_index': current_index + 1
    }


def should_continue_slides(state: PptState):
    """Check if we need to generate more slides"""
    current_index = state.get('current_slide_index', 0)
    total_slides = len(state.get('outline', {}).get('slides', []))
    
    if current_index < total_slides:
        return "continue"
    else:
        return "end"


# Build the workflow graph
tool_node = ToolNode(tools)

workflow = StateGraph(PptState)

# Add nodes
workflow.add_node("generate_outline", generate_outline_node)
workflow.add_node("generate_slide_detail", generate_slide_detail_node)
workflow.add_node("chat_node", chat_node)
workflow.add_node("tools", tool_node)

# Define the flow
workflow.add_edge(START, "generate_outline")
workflow.add_edge("generate_outline", "generate_slide_detail")

# Conditional edge for continuing slides
workflow.add_conditional_edges(
    "generate_slide_detail",
    should_continue_slides,
    {
        "continue": "generate_slide_detail",
        "end": END
    }
)

# Compile the workflow
ppt_generator = workflow.compile()


# Usage example
def generate_presentation(topic: str, num_slides: int = 5):
    """Main function to generate a complete presentation"""
    
    print(f"🎯 Starting PPT generation on: {topic}")
    print(f"📊 Number of slides: {num_slides}\n")
    
    initial_state = {
        'messages': [],
        'outline': {},
        'detailed_slides': [],
        'current_slide_index': 0,
        'topic': topic,
        'num_slides': num_slides
    }
    
    result = ppt_generator.invoke(initial_state)
    
    print("=" * 60)
    print("STEP 1: PRESENTATION OUTLINE")
    print("=" * 60)
    outline = result['outline']
    print(f"\nTitle: {outline.get('title', 'N/A')}")
    print(f"Total Slides: {outline.get('total_slides', 'N/A')}\n")
    
    for slide in outline.get('slides', []):
        print(f"Slide {slide['slide_number']}: {slide['slide_title']}")
        print(f"  Type: {slide['content_type']}")
        print(f"  Key Points: {', '.join(slide['key_points'])}\n")
    
    print("\n" + "=" * 60)
    print("STEP 2: DETAILED SLIDE CONTENT")
    print("=" * 60)
    
    for slide_detail in result['detailed_slides']:
        print(f"\n{'=' * 60}")
        print(f"SLIDE {slide_detail.get('slide_number', 'N/A')}: {slide_detail.get('slide_title', 'N/A')}")
        print(f"{'=' * 60}")
        
        if 'detailed_content' in slide_detail:
            for content in slide_detail['detailed_content']:
                print(f"\n📍 {content.get('key_point', 'N/A')}")
                print(f"   {content.get('explanation', 'N/A')}")
                if 'visual_suggestion' in content:
                    print(f"   💡 Visual: {content['visual_suggestion']}")
        
        if 'speaker_notes' in slide_detail:
            print(f"\n📝 Speaker Notes: {slide_detail['speaker_notes']}")
        
        if 'raw_response' in slide_detail:
            print(f"\n📄 Content:\n{slide_detail['raw_response']}")
    
    return result


# Run the example
if __name__ == "__main__":
    result = generate_presentation(
        topic="Photosynthesis in Plants",
        num_slides=5
    )
    
    # You can also save the result to a JSON file
    with open('presentation_output.json', 'w') as f:
        json.dump({
            'outline': result['outline'],
            'detailed_slides': result['detailed_slides']
        }, f, indent=2)
    
    print("\n✅ Presentation generated and saved to 'presentation_output.json'")

C:\Users\kaushal\AppData\Local\Temp\ipykernel_19240\4288737946.py:15: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-tavily package and should be used instead. To use it run `pip install -U :class:`~langchain-tavily` and import as `from :class:`~langchain_tavily import TavilySearch``.
  searchTool = TavilySearchResults(max_results=3)


🎯 Starting PPT generation on: Photosynthesis in Plants
📊 Number of slides: 5

STEP 1: PRESENTATION OUTLINE

Title: Photosynthesis in Plants
Total Slides: 5

Slide 1: Introduction to Photosynthesis
  Type: introduction
  Key Points: Definition of Photosynthesis, Importance of Photosynthesis, Key Components Involved

Slide 2: The Process of Photosynthesis
  Type: explanation
  Key Points: Light Absorption, Carbon Dioxide Uptake, Water Splitting and Electron Transport

Slide 3: Stages of Photosynthesis
  Type: comparison
  Key Points: Light-Dependent Reactions, Calvin Cycle (Light-Independent Reactions), ATP and NADPH Utilization

Slide 4: Products of Photosynthesis
  Type: explanation
  Key Points: Glucose and Oxygen, Role in Plant Growth and Energy Storage, Impact on Ecosystems

Slide 5: Conclusion and Applications
  Type: conclusion
  Key Points: Summary of Key Points, Applications in Biotechnology and Agriculture, Future Research Directions


STEP 2: DETAILED SLIDE CONTENT

SLIDE 1: I